# Exercise 2 — send_alert

`send_alert` is the notification layer: it prints to stdout when no webhook is configured, and POSTs to a webhook URL when one is provided. For paper trading, the stdout path is sufficient. The webhook path enables Slack, Discord, or any other notification system with a single URL.

In [ ]:
import pandas as pd, math, datetime, pathlib, tempfile

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })

def send_alert(message, webhook_url=None):
    """Send an alert via webhook or stdout.

    If webhook_url is None:
      print(f"[ALERT] {message}") and return True.

    If webhook_url is provided:
      POST JSON {"text": message} with requests.post(webhook_url, ..., timeout=5).
      Return resp.ok on success; return False on any exception.

    Returns:
        bool
    """
    # TODO: ~8 lines
    return False


### Checks

In [ ]:
from unittest.mock import patch
checks = 0

# 1 — webhook_url=None returns True
try:
    with patch("builtins.print"):
        result = send_alert("Test message", webhook_url=None)
    assert result is True
    checks += 1; print("✅ 1 send_alert(webhook=None) returns True")
except Exception as e:
    print("❌ 1:", e)

# 2 — webhook_url=None calls print with [ALERT] prefix
try:
    with patch("builtins.print") as mock_print:
        send_alert("Hello world", webhook_url=None)
    assert mock_print.called, "send_alert should call print"
    call_str = " ".join(str(a) for a in mock_print.call_args[0])
    assert "[ALERT]" in call_str, f"expected [ALERT] in: {call_str}"
    assert "Hello world" in call_str
    checks += 1; print("✅ 2 send_alert calls print with [ALERT] + message")
except Exception as e:
    print("❌ 2:", e)

# 3 — empty message: still returns True
try:
    with patch("builtins.print"):
        result = send_alert("", webhook_url=None)
    assert result is True
    checks += 1; print("✅ 3 empty message with webhook=None still returns True")
except Exception as e:
    print("❌ 3:", e)

# 4 — bad webhook URL: returns False (exception caught)
try:
    result = send_alert("Test", webhook_url="http://localhost:0/bad")
    assert result is False, f"bad URL should return False, got {result}"
    checks += 1; print("✅ 4 bad webhook URL → False (exception caught gracefully)")
except Exception as e:
    print("❌ 4:", e)

# 5 — two consecutive alerts: print called twice, with distinct messages
try:
    calls = []
    with patch("builtins.print", side_effect=lambda *a, **kw: calls.extend(a)):
        send_alert("Alert A", None)
        send_alert("Alert B", None)
    assert len(calls) == 2, f"expected 2 print calls, got {len(calls)}"
    combined = " ".join(str(c) for c in calls)
    assert "Alert A" in combined and "Alert B" in combined
    checks += 1; print("✅ 5 two alerts → print called twice with distinct messages")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
